In [0]:
import requests
import pandas as pd

# Fetch climate anomaly dataset (Global Monthly Mean)
# NASA GISS Surface Temperature Analysis (GISTEMP v4)
url = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"

# Add headers to avoid 403 Forbidden error
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response = requests.get(url, headers=headers)

# Save raw file to  Unity Catalog Volume
volume_path = "/Volumes/climate_portfolio/global_warming/raw_landing/nasa_global_temp.csv"
with open(volume_path, "wb") as f:
    f.write(response.content)

# Read into PySpark 
# (NASA data uses a few header rows, we skip or clean them)
# Skip first row (metadata) and use second row as header
raw_df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("skipRows", 1)
          .csv(volume_path))

# Clean column names to remove invalid characters for Delta Lake
for col_name in raw_df.columns:
    clean_name = col_name.replace(".", "_").replace(",", "_").replace(";", "_").replace("{", "_").replace("}", "_").replace("(", "_").replace(")", "_").replace("\n", "_").replace("\t", "_").replace("=", "_").replace(" ", "_").replace("<", "_").replace(">", "_").replace("!", "_")
    raw_df = raw_df.withColumnRenamed(col_name, clean_name)

# Drop existing table to avoid schema mismatch
spark.sql("DROP TABLE IF EXISTS climate_portfolio.global_warming.bronze_global_temp")

# Save to Bronze Delta Table
raw_df.write.mode("overwrite").saveAsTable("climate_portfolio.global_warming.bronze_global_temp")
print("Bronze layer created successfully!")


In [0]:
from pyspark.sql.functions import col, expr

# Read 
bronze_df = spark.table("climate_portfolio.global_warming.bronze_global_temp")

# Cast all month columns to string to handle mixed types before unpivot
for month_col in ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]:
    bronze_df = bronze_df.withColumn(month_col, col(month_col).cast("string"))

# Unpivot months from wide columns into a clean 'Month' and 'Anomaly' structure
# Filter out non-numeric artifact rows from the raw file
silver_df = (bronze_df
             .filter(col("Year").cast("int").isNotNull())
             .unpivot(["Year"], ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], "Month", "Anomaly")
             .select(
                 col("Year").cast("int").alias("year"),
                 col("Month").alias("month"),
                 expr("try_cast(Anomaly as double)").alias("temperature_anomaly_celsius")
             )
             .filter(col("temperature_anomaly_celsius").isNotNull()))

# Save to Silver Delta Table
silver_df.write.mode("overwrite").saveAsTable("climate_portfolio.global_warming.silver_clean_anomalies")
print("Silver layer structured and long-form format established!")


In [0]:
%sql
CREATE OR REPLACE TABLE climate_portfolio.global_warming.gold_decadal_trends AS
SELECT 
    year,
    AVG(temperature_anomaly_celsius) AS yearly_avg_anomaly,
    -- Compute a 10-year rolling average window over time
    AVG(AVG(temperature_anomaly_celsius)) OVER (
        ORDER BY year 
        ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
    ) AS rolling_10_year_anomaly
FROM climate_portfolio.global_warming.silver_clean_anomalies
GROUP BY year;
